In [1]:
"""Ячейка 1 — импорты и переменные окружения.

`load_dotenv()` подтягивает `LLM_API_KEY` из `.env`, без него RAGAS-метрики
(Ячейка 5) не смогут поднять LLM-клиент. `json` и `Path` нужны только в
Ячейке 6 для записи итогового файла с метриками.
"""
import json
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
"""Ячейка 2 — golden-датасет из 18 эталонных вопросов.

Структура: 13 английских и 4 русских вопроса по всем 10 разделам корпуса
(linear_model / tree / model_evaluation + ensemble / cross_validation /
preprocessing / compose / grid_search / impute / feature_selection) и
1 meta-вопрос про `about.md`. Поле `ground_truth_url_keywords` нужно для
Recall@k в Ячейке 4: считаем hit, если URL retrieved-чанка содержит
ожидаемое ключевое слово.
"""
GOLDEN = [
    # --- 7 EN in-corpus вопросов ---
    {
        "question": "How does Ridge regression handle multicollinearity?",
        "ground_truth": "Ridge adds an L2 penalty alpha * sum(w_i^2) to the loss, "
                        "which shrinks correlated coefficients toward each other.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What does the alpha parameter control in Ridge?",
        "ground_truth": "Alpha controls regularization strength; larger alpha means "
                        "stronger penalty and smaller coefficients.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What is the difference between Lasso and Ridge?",
        "ground_truth": "Lasso uses L1 penalty which can zero out coefficients (feature "
                        "selection); Ridge uses L2 which shrinks but never zeroes.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What does min_samples_leaf control in a decision tree?",
        "ground_truth": "min_samples_leaf is the minimum number of samples required to be "
                        "at a leaf node; higher values prevent overfitting by limiting depth.",
        "ground_truth_url_keywords": ["tree"],
    },
    {
        "question": "When does a decision tree overfit?",
        "ground_truth": "Trees overfit when grown too deep without min_samples_leaf or "
                        "min_samples_split constraints, memorising training noise.",
        "ground_truth_url_keywords": ["tree"],
    },
    {
        "question": "What is the formula for precision?",
        "ground_truth": "precision = TP / (TP + FP). Fraction of positive predictions that "
                        "are actually positive.",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    {
        "question": "When is recall more important than precision?",
        "ground_truth": "Recall matters most when missing positives is costly: cancer "
                        "screening, fraud detection, anything where false negatives are "
                        "more harmful than false positives.",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    # --- 2 RU вопроса (тест мультиязычного retrieval) ---
    {
        "question": "Что такое L2-регуляризация?",
        "ground_truth": "L2-регуляризация добавляет к функции потерь штраф, "
                        "пропорциональный сумме квадратов коэффициентов модели. "
                        "Используется в Ridge.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "Что такое F1-мера?",
        "ground_truth": "F1 — гармоническое среднее precision и recall, "
                        "F1 = 2 * precision * recall / (precision + recall).",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    # --- 8 вопросов по расширенному корпусу ---
    {
        "question": "How does Random Forest reduce variance compared to a single decision tree?",
        "ground_truth": "Random Forest averages many decorrelated trees trained on bootstrap "
                        "samples, so individual tree variance cancels out.",
        "ground_truth_url_keywords": ["ensemble"],
    },
    {
        "question": "What is the difference between Gradient Boosting and Random Forest?",
        "ground_truth": "Gradient Boosting fits trees sequentially, each correcting the previous "
                        "ensemble's errors; Random Forest trains trees independently and averages them.",
        "ground_truth_url_keywords": ["ensemble"],
    },
    {
        "question": "Когда нужен StratifiedKFold вместо обычного KFold?",
        "ground_truth": "StratifiedKFold сохраняет доли классов в каждом фолде — это критично "
                        "для несбалансированных классов и маленьких выборок.",
        "ground_truth_url_keywords": ["cross_validation"],
    },
    {
        "question": "Why is feature scaling important for regularised linear models?",
        "ground_truth": "Регуляризация штрафует величину коэффициентов, поэтому неотскейленные "
                        "признаки штрафуются неравномерно; scaling делает alpha осмысленной.",
        "ground_truth_url_keywords": ["preprocessing"],
    },
    {
        "question": "How does a Pipeline help avoid data leakage?",
        "ground_truth": "Pipeline вызывает fit только на train-части и применяет уже обученные "
                        "трансформеры к тесту, поэтому статистики вроде среднего скалера не видят тест.",
        "ground_truth_url_keywords": ["compose"],
    },
    {
        "question": "Чем RandomizedSearchCV отличается от GridSearchCV?",
        "ground_truth": "GridSearchCV перебирает все комбинации параметров, RandomizedSearchCV — "
                        "заданное число случайных комбинаций; на больших пространствах второй быстрее.",
        "ground_truth_url_keywords": ["grid_search"],
    },
    {
        "question": "How does SimpleImputer fill missing values?",
        "ground_truth": "SimpleImputer заменяет пропуски статистикой, посчитанной на train "
                        "(mean, median, most_frequent или константа), и умеет добавлять индикаторы пропусков.",
        "ground_truth_url_keywords": ["impute"],
    },
    {
        "question": "What does SelectFromModel do?",
        "ground_truth": "SelectFromModel оставляет признаки, чья важность или модуль коэффициента "
                        "выше порога, вычисленного по обученному estimator.",
        "ground_truth_url_keywords": ["feature_selection"],
    },
    # --- 1 meta-вопрос (тест about.md) ---
    {
        "question": "Что ты умеешь?",
        "ground_truth": "Отвечаю на вопросы по трём разделам scikit-learn: линейные "
                        "модели, деревья решений, метрики качества.",
        "ground_truth_url_keywords": ["about.md", "local"],
    },
]

In [3]:
"""Ячейка 3 — прямое подключение к RAG-pipeline без HTTP-слоя.

`build_rag_chain()` поднимает retriever (Qdrant + embedder) и LCEL-цепочку
в этом же процессе. Прямой вызов даёт чистые тайминги и доступ к
`retrieved_contexts`, которые понадобятся RAGAS в Ячейке 5.
"""
import sys
sys.path.insert(0, r'C:\Users\DW\Desktop\mentoring\task-api')
from app.core.config import settings
from app.rag.chain import build_rag_chain
chain, retriever = build_rag_chain()

C:\Users\DW\anaconda3\envs\task-api\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\DW\anaconda3\envs\task-api\Lib\site-packages\qdrant_client\qdrant_remote.py:275: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7959.67it/s]

In [4]:
"""Ячейка 4 — Recall@k для retriever'а.

Прогоняем каждый golden-вопрос через retriever, забираем URL'ы топ-k чанков
и считаем hit по keyword-совпадению. Дополнительно копим
`retrieved_contexts` — они уйдут в RAGAS на следующем шаге.
"""
def url_match(returned_urls: list[str], expected_keywords: list[str]) -> bool:
    """Помечает retrieval как hit, если хоть один URL содержит ожидаемое ключевое слово.

    Используется в Recall@k: keyword-matching на уровне модуля sklearn
    (`linear_model` / `tree` / `model_evaluation`) — этого достаточно
    для понимания «не промазал ли retriever мимо темы целиком».
    """
    if not expected_keywords:  # ловушка под OOC-вопросы, если решите добавлять
        return not returned_urls
    return any(
        any(kw.lower() in url.lower() for kw in expected_keywords)
        for url in returned_urls
    )

results = []
for item in GOLDEN:
    docs = retriever.invoke(item["question"])
    urls = [doc.metadata.get("source", "") for doc in docs]
    results.append({
        "question": item["question"],
        "ground_truth": item["ground_truth"],
        "retrieved_urls": urls,
        "retrieved_contexts": [doc.page_content for doc in docs],
        "retriever_hit": url_match(urls, item["ground_truth_url_keywords"]),
    })

hits = sum(1 for r in results if r["retriever_hit"])
recall_at_k = hits / len(results)
print(f"Retriever Recall@{settings.top_k}: {recall_at_k:.3f}  ({hits}/{len(results)})")

Retriever Recall@4: 1.000  (18/18)


In [5]:
"""Ячейка 5 — generation-метрики через RAGAS.

RAGAS 0.2.x требует Dataset со строго заданными именами полей
(`user_input` / `response` / `retrieved_contexts` / `reference`). LLM
и эмбеддер оборачиваем в `LangchainLLMWrapper` и
`LangchainEmbeddingsWrapper` — без этого `evaluate()` не поймёт, как
через них ходить. Считаем Faithfulness (опора на контекст) и
ResponseRelevancy (релевантность ответа вопросу).
"""
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import Faithfulness, ResponseRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_huggingface import HuggingFaceEmbeddings

from app.llm import get_llm

# Прогоняем chain и собираем ответы
for r in results:
    r["response"] = chain.invoke(r["question"])

# RAGAS 0.2.x требует именно эти имена полей в Dataset:
# user_input / response / retrieved_contexts / reference
ragas_data = Dataset.from_list([
    {
        "user_input": r["question"],
        "response": r["response"],
        "retrieved_contexts": r["retrieved_contexts"],
        "reference": r["ground_truth"],
    }
    for r in results
])

# LLM и эмбеддер для RAGAS оборачиваются в специальные wrapper'ы
llm = LangchainLLMWrapper(get_llm())
emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name="intfloat/multilingual-e5-small",
        encode_kwargs={"normalize_embeddings": True},
    )
)

ragas_scores = evaluate(
    dataset=ragas_data,
    metrics=[
        Faithfulness(llm=llm),
        ResponseRelevancy(llm=llm, embeddings=emb),
    ],
)
print(ragas_scores)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7959.74it/s]

Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

Evaluating:   3%|▎         | 1/36 [00:03<02:14,  3.84s/it]

Evaluating:   6%|▌         | 2/36 [00:05<01:19,  2.34s/it]

...

Evaluating:  94%|█████████▍| 34/36 [00:47<00:09,  4.90s/it]

Evaluating:  97%|█████████▋| 35/36 [00:54<00:05,  5.53s/it]

Evaluating: 100%|██████████| 36/36 [01:39<00:00, 17.49s/it]

Evaluating: 100%|██████████| 36/36 [01:39<00:00,  2.78s/it]

{'faithfulness': 0.8778, 'answer_relevancy': 0.9479}


In [6]:
import os
# Если не корень проекта, измени на нужную
os.chdir(r"C:\Users\DW\Desktop\mentoring\task-api")
"""Ячейка 6 — сохраняем сводку метрик в JSON.

Берём mean по каждой RAGAS-метрике, добавляем retriever-Recall@k и
метаданные прогона (модель, embedder, top_k). Файл `rag_metrics.json`
цитируется в README + входит в будущую легенду на собесе.
"""
# RAGAS 0.2.x возвращает EvaluationResult — берём .to_pandas() для агрегации
df = ragas_scores.to_pandas()
gen_scores = {
    "faithfulness": float(df["faithfulness"].mean()),
    "answer_relevancy": float(df["answer_relevancy"].mean()),
}

metrics = {
    "n_questions": len(GOLDEN),
    "model": settings.llm_model,
    "embedding_model": settings.embedding_model,
    "top_k": settings.top_k,
    "retriever": {
        f"recall_at_{settings.top_k}": recall_at_k,
        "hits": hits,
    },
    "generation": gen_scores,
}
Path("notebooks/rag_metrics.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False)
)
print(json.dumps(metrics, indent=2, ensure_ascii=False))

{
  "n_questions": 18,
  "model": "gpt://b1gqeb7j1sefk9u2jehe/yandexgpt-5-lite",
  "embedding_model": "intfloat/multilingual-e5-small",
  "top_k": 4,
  "retriever": {
    "recall_at_4": 1.0,
    "hits": 18
  },
  "generation": {
    "faithfulness": 0.8777777777777778,
    "answer_relevancy": 0.9479282812431725
  }
}
